数据存放格式要求：（tile的名称命名没有要求）
HDdata/                     <-- base_path (主目录)
├── OV/                     <-- 子目录 (代表不同的癌症类型或样本)
│   ├── tile_0001.pt        <-- PyTorch 数据文件
│   ├── tile_0002.pt
│   └── ...
├── CESC/
│   ├── tile_0001.pt
│   ├── tile_0002.pt
│   └── ...
├── PRAD/
│   ├── tile_0001.pt
│   └── ...
├── NSCLC/
│   ├── tile_0001.pt
│   └── ...
└── COAD/
    ├── tile_0001.pt
    └── ...

In [ ]:
import os
import glob
import torch
from torch.utils.data import Dataset, DataLoader

# ==========================================
# 全局参数设置
# ==========================================
# SRC_DIR: 原始数据目录，包含多个子目录，每个子目录对应一个样本
# DST_DIR: 处理后数据的保存目录
# MIN_NUC: 中心区域细胞核数量的最小阈值，只有满足条件的样本才会被保留

SRC_DIR = '/root/autodl-tmp/HDdata'
DST_DIR = '/root/autodl-tmp/HD_dataset'
MIN_NUC = 5
PATCH_SIZE = 128
OVERLAP = 16

In [ ]:
# ==========================================
# 1. 检查并注解中心区域细胞核数量
# ==========================================
def check_and_annotate(src_dir, patch_size, overlap):
    margin = overlap // 2
    min_bound, max_bound = margin, patch_size - margin
    
    subdirs = [os.path.join(src_dir, d) for d in os.listdir(src_dir) if os.path.isdir(os.path.join(src_dir, d))]
    
    for subdir in subdirs:
        files = glob.glob(os.path.join(subdir, "*.pt"))
        if not files:
            continue
            
        needs_annotation = False
        for f in files[:10]:
            data = torch.load(f)
            if "center_nuclei_count" not in data:
                needs_annotation = True
                break
                
        if not needs_annotation:
            print(f"[{os.path.basename(subdir)}] Already annotated. Skipping.")
            continue
            
        print(f"[{os.path.basename(subdir)}] Annotating...")
        for filepath in files:
            tile_data = torch.load(filepath)
            nuclei_tensor = tile_data["input_nuclei"]
            
            if nuclei_tensor._nnz() == 0:
                count = 0
            else:
                indices = nuclei_tensor.indices()
                values = nuclei_tensor.values()
                
                spatial_mask = (indices[0] >= min_bound) & (indices[0] < max_bound) & \
                               (indices[1] >= min_bound) & (indices[1] < max_bound)
                
                count = torch.unique(values[spatial_mask]).numel()
            
            tile_data["center_nuclei_count"] = count
            torch.save(tile_data, filepath)

# ==========================================
# 2. 离线筛选与构建新数据集 (增量处理 + 统计)
# ==========================================
def build_filtered_dataset(src_dir, dst_dir, min_nuc):
    subdirs = [os.path.join(src_dir, d) for d in os.listdir(src_dir) if os.path.isdir(os.path.join(src_dir, d))]
    
    total_raw = 0
    total_load = 0
    
    for subdir in subdirs:
        subdir_name = os.path.basename(subdir)
        file_paths = glob.glob(os.path.join(subdir, "*.pt"))
        
        raw_count = len(file_paths)
        if raw_count == 0:
            continue
            
        total_raw += raw_count
        load_count = 0
        
        save_dir = os.path.join(dst_dir, subdir_name)
        os.makedirs(save_dir, exist_ok=True)
        
        for path in file_paths:
            filename = os.path.basename(path)
            save_path = os.path.join(save_dir, filename)
            
            # 增量检查：如果目标文件已存在，直接计入 load_count
            if os.path.exists(save_path):
                load_count += 1
                continue
                
            tile_data = torch.load(path)
            if tile_data.get("center_nuclei_count", 0) < min_nuc:
                continue
                
            processed_data = {
                "input_expr": tile_data["input_expr"].to_dense().permute(2, 0, 1),
                "input_nuclei": tile_data["input_nuclei"].to_dense().permute(2, 0, 1).float(),
                "target_expr": tile_data["target_expr"].to_dense().permute(2, 0, 1),
                "target_cell_id": tile_data["target_cell_id"].to_dense().permute(2, 0, 1).float()
            }
            
            torch.save(processed_data, save_path)
            load_count += 1
            
        total_load += load_count
        print(f"{subdir_name}: raw tile: {raw_count}, load tile: {load_count}")
        
    print("-" * 40)
    print(f"Total: raw tile: {total_raw}, load tile: {total_load}")

# ==========================================
# 3. Dataset 类
# ==========================================
class SpatialTranscriptomicsDataset(Dataset):
    def __init__(self, data_dir: str):
        self.file_paths = glob.glob(os.path.join(data_dir, "*", "*.pt"))

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        return torch.load(self.file_paths[idx], weights_only=True)

# ==========================================
# 4. 主执行流程
# ==========================================
if __name__ == "__main__":
    check_and_annotate(src_dir=SRC_DIR, patch_size=PATCH_SIZE, overlap=OVERLAP)
    
    build_filtered_dataset(src_dir=SRC_DIR, dst_dir=DST_DIR, min_nuc=MIN_NUC)
    
    dataset = SpatialTranscriptomicsDataset(data_dir=DST_DIR)
    print(f"Total samples for training: {len(dataset)}")
    
    if len(dataset) > 0:
        dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=4, pin_memory=True)
        for batch_data in dataloader:
            print(f"Batch input_expr shape: {batch_data['input_expr'].shape}")
            break